# 05. Kiểm định thống kê (C1 - C3)

Notebook này kiểm định các giả thuyết về tỷ lệ chỉnh sửa code trong 30 ngày (`change_30d`), mục đích sửa lỗi (`bug_fix`) và kích thước đoạn code. Tất cả kiểm định chính đều tính sai số chuẩn theo cụm `repo` vì các khối code trong cùng một repo không độc lập với nhau.

Import thư viện, tải dữ liệu đã xử lý và chuẩn bị tập dữ liệu đủ điều kiện theo dõi 30 ngày (`df_30d`).

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

DATA_PATH = Path("../data/processed/commits_features.csv")
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)
for col in ["commit_date", "change_commit_date"]:
    df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

df["has_recorded_change"] = df["has_recorded_change"].astype(int)
df["days_to_modification"] = pd.to_numeric(df["days_to_modification"], errors="coerce")
df["log_code_lines"] = np.log1p(df["code_lines"])

df_30d = df[df["change_30d"].notna()].copy()
df_30d["change_30d"] = df_30d["change_30d"].astype(int)

ALPHA = 0.05

def cluster_ols(formula, data):
    groups = pd.factorize(data["repo"])[0]
    return smf.ols(formula, data=data).fit(cov_type="cluster", cov_kwds={"groups": groups})

print(f"Total blocks: {len(df):,} ({df['repo'].nunique():,} repos)")
print(f"30-day eligible blocks: {len(df_30d):,} ({df_30d['repo'].nunique():,} repos)")

Total blocks: 21,287 (12,087 repos)
30-day eligible blocks: 19,554 (11,234 repos)


## C1. Kiểm định giả thuyết

Định nghĩa các hàm kiểm định Chi-square (kèm p-value theo cụm repo và Cramer's V) và Mann-Whitney U (kèm hồi quy trên `log1p` theo cụm repo).

In [2]:
def cramers_v(table, chi2):
    n = table.to_numpy().sum()
    r, k = table.shape
    return np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1)))

def chi_square_cluster(name, data, row, target):
    table = pd.crosstab(data[row], data[target])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    p_cluster = cluster_ols(f"{target} ~ C({row})", data).f_pvalue
    return {
        "question": name,
        "test": "Chi-square + Cluster Wald",
        "n": len(data),
        "statistic": chi2,
        "p_value_raw": p,
        "p_value_cluster": p_cluster,
        "effect_size": cramers_v(table, chi2),
        "significant_0.05": p_cluster < ALPHA,
    }

def numeric_group_test(name, data, x_col, group_col):
    sub = data[[x_col, group_col, "repo"]].dropna()
    a = sub.loc[sub[group_col] == 0, x_col]
    b = sub.loc[sub[group_col] == 1, x_col]
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rbc = 2 * u / (len(a) * len(b)) - 1
    p_cluster = cluster_ols(f"np.log1p({x_col}) ~ {group_col}", sub).pvalues[group_col]
    return {
        "question": name,
        "test": "Mann-Whitney U + Cluster log-OLS",
        "n": len(sub),
        "statistic": u,
        "p_value_raw": p,
        "p_value_cluster": p_cluster,
        "effect_size": rbc,
        "significant_0.05": p_cluster < ALPHA,
    }

Chạy các kiểm định C1 cho `change_30d` (theo ngôn ngữ, năm, công cụ AI, vai trò file, độ dài code) và kiểm tra tỷ lệ sửa lỗi (`bug_fix`) nếu có dữ liệu commit message.

In [3]:
c1_results = [
    chi_square_cluster("C1.1 language x change_30d", df_30d, "language", "change_30d"),
    chi_square_cluster("C1.2 year x change_30d", df_30d, "year", "change_30d"),
    chi_square_cluster("C1.3 ai_tool x change_30d", df_30d, "ai_tool", "change_30d"),
    chi_square_cluster("C1.4 file_role x change_30d", df_30d, "file_role", "change_30d"),
    numeric_group_test("C1.5 code_lines by change_30d", df_30d, "code_lines", "change_30d"),
]

days_df = df[df["days_to_modification"].notna()].assign(is_python=(df["language"] == "python").astype(int))
c1_results.append(numeric_group_test("C1.6 days_to_modification: Python vs JS", days_df, "days_to_modification", "is_python"))

changed = df[df["has_recorded_change"] == 1].copy()
if len(set(changed["change_intent"].unique()) - {"other_change", "no_recorded_change"}) > 0:
    changed["is_bug_fix"] = (changed["change_intent"] == "bug_fix").astype(int)
    c1_results.append(chi_square_cluster("C1.7 file_role x is_bug_fix", changed, "file_role", "is_bug_fix"))
    c1_results.append(chi_square_cluster("C1.8 ai_tool x is_bug_fix", changed, "ai_tool", "is_bug_fix"))

c1_table = pd.DataFrame(c1_results)
display(c1_table)
c1_table.to_csv(REPORT_DIR / "statistical_c1_hypothesis_tests.csv", index=False)

,question,test,n,statistic,p_value_raw,p_value_cluster,effect_size,significant_0.05
0,C1.1 language x change_30d,Chi-square + Cluster Wald,19554,1.285345e+01,3.368582e-04,5.547673e-02,0.025638,False
1,C1.2 year x change_30d,Chi-square + Cluster Wald,19554,6.011806e+01,2.739787e-12,4.612848e-05,0.055448,True
2,C1.3 ai_tool x change_30d,Chi-square + Cluster Wald,19554,3.184103e+01,1.750325e-05,1.426470e-02,0.040353,True
3,C1.4 file_role x change_30d,Chi-square + Cluster Wald,19554,2.879022e+01,8.623141e-06,2.361773e-02,0.038371,True
4,C1.5 code_lines by change_30d,Mann-Whitney U + Cluster log-OLS,19554,2.823187e+07,2.574461e-64,8.773667e-27,-0.166111,True
5,C1.6 days_to_modification: Python vs JS,Mann-Whitney U + Cluster log-OLS,5714,2.731078e+06,9.315689e-06,3.225026e-01,-0.079422,False
6,C1.7 file_role x is_bug_fix,Chi-square + Cluster Wald,7445,1.870846e+01,8.966604e-04,7.516490e-02,0.050129,False
7,C1.8 ai_tool x is_bug_fix,Chi-square + Cluster Wald,7445,4.900803e+02,1.151856e-102,2.653301e-18,0.256567,True


## C2. So sánh A/B theo cụm repo

So sánh chênh lệch tỷ lệ `change_30d` (và `is_bug_fix` nếu có) giữa các cặp nhóm chính kèm khoảng tin cậy 95% theo cụm repo và chỉ số Cohen's h.

In [4]:
def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

def ab_test(name, data, target, mask_a, mask_b):
    d = data[mask_a | mask_b].copy()
    d["group_a"] = mask_a.loc[d.index].astype(int)
    res = cluster_ols(f"{target} ~ group_a", d)
    pa = d.loc[d["group_a"] == 1, target].mean()
    pb = d.loc[d["group_a"] == 0, target].mean()
    lo, hi = res.conf_int().loc["group_a"]
    return {
        "comparison": name,
        "target": target,
        "group_a_n": int(d["group_a"].sum()),
        "group_b_n": int((d["group_a"] == 0).sum()),
        "group_a_rate": pa,
        "group_b_rate": pb,
        "diff_a_minus_b": pa - pb,
        "diff_95CI_cluster": f"[{lo:.4f}, {hi:.4f}]",
        "cohens_h": cohens_h(pa, pb),
        "p_value_cluster": res.pvalues["group_a"],
        "significant_0.05": res.pvalues["group_a"] < ALPHA,
    }

ab_results = [
    ab_test("C2.1 Python vs JS", df_30d, "change_30d", df_30d["language"] == "python", df_30d["language"] == "js"),
    ab_test("C2.2 Weekend vs weekday", df_30d, "change_30d", df_30d["is_weekend"] == True, df_30d["is_weekend"] == False),
    ab_test("C2.3 core_source vs notebook_or_script", df_30d, "change_30d", df_30d["file_role"] == "core_source", df_30d["file_role"] == "notebook_or_script"),
    ab_test("C2.4 written vs assisted", df_30d, "change_30d", df_30d["ai_verb"] == "written", df_30d["ai_verb"] == "assisted"),
]

if "is_bug_fix" in changed.columns:
    ab_results.append(
        ab_test("C2.5 Bug-fix rate: core_source vs test", changed, "is_bug_fix", changed["file_role"] == "core_source", changed["file_role"] == "test")
    )

ab_table = pd.DataFrame(ab_results)
display(ab_table)
ab_table.to_csv(REPORT_DIR / "statistical_c2_ab_tests.csv", index=False)

,comparison,target,group_a_n,group_b_n,group_a_rate,group_b_rate,diff_a_minus_b,diff_95CI_cluster,cohens_h,p_value_cluster,significant_0.05
0,C2.1 Python vs JS,change_30d,14386,5168,0.236480,0.211881,0.024599,"[-0.0006, 0.0498]",0.059005,0.055462,False
1,C2.2 Weekend vs weekday,change_30d,4430,15124,0.240858,0.226792,0.014066,"[-0.0073, 0.0355]",0.033236,0.197389,False
2,C2.3 core_source vs notebook_or_script,change_30d,14504,1876,0.236900,0.193497,0.043403,"[0.0123, 0.0745]",0.105735,0.006193,True
3,C2.4 written vs assisted,change_30d,16856,2698,0.232202,0.216086,0.016116,"[-0.0096, 0.0419]",0.038652,0.219915,False
4,C2.5 Bug-fix rate: core_source vs test,is_bug_fix,5524,522,0.255250,0.327586,-0.072336,"[-0.1868, 0.0422]",-0.159462,0.215618,False


## C3. Khoảng tin cậy Bootstrap theo cụm repo

Ước lượng khoảng tin cậy 95% cho trung vị số ngày đến lần sửa đầu tiên và tỷ lệ chỉnh sửa trong 30 ngày theo từng năm (có kiểm tra độ nhạy khi loại bỏ repo `conradhuebler/ALIMA`).

In [5]:
rng = np.random.default_rng(42)

def bootstrap_stat(values, statistic=np.median, n_boot=5000):
    values = np.asarray(values, dtype=float)
    estimates = np.empty(n_boot)
    for i in range(n_boot):
        estimates[i] = statistic(rng.choice(values, size=len(values), replace=True))
    return float(statistic(values)), float(np.quantile(estimates, 0.025)), float(np.quantile(estimates, 0.975))

def cluster_rate(d, col, n_boot=5000):
    g = d.groupby("repo")[col].agg(["sum", "size"]).to_numpy(dtype=float)
    idx = rng.integers(0, len(g), size=(n_boot, len(g)))
    boot = g[idx, 0].sum(axis=1) / g[idx, 1].sum(axis=1)
    return float(g[:, 0].sum() / g[:, 1].sum()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))

valid_days = df["days_to_modification"].dropna().to_numpy()
median, lo, hi = bootstrap_stat(valid_days, np.median)
bootstrap_median = pd.DataFrame([{
    "metric": "median_days_to_modification",
    "n": len(valid_days),
    "estimate": median,
    "ci_95_lower": lo,
    "ci_95_upper": hi,
}])
display(bootstrap_median)
bootstrap_median.to_csv(REPORT_DIR / "statistical_c3_median_bootstrap.csv", index=False)

OUTLIER_REPO = "conradhuebler/ALIMA"
year_rows = []
for year, group in df_30d.groupby("year", dropna=True):
    rate_30, lo_30, hi_30 = cluster_rate(group, "change_30d")
    no_alima = group[group["repo"] != OUTLIER_REPO]
    rate_x, lo_x, hi_x = cluster_rate(no_alima, "change_30d")
    year_rows.append({
        "year": int(year),
        "n_30d": len(group),
        "n_repos": group["repo"].nunique(),
        "change_rate_30d": rate_30,
        "ci_30d_lower": lo_30,
        "ci_30d_upper": hi_30,
        "change_rate_30d_no_alima": rate_x,
        "no_alima_lower": lo_x,
        "no_alima_upper": hi_x,
    })

bootstrap_year = pd.DataFrame(year_rows).sort_values("year")
display(bootstrap_year)
bootstrap_year.to_csv(REPORT_DIR / "statistical_c3_year_bootstrap.csv", index=False)

,metric,n,estimate,ci_95_lower,ci_95_upper
0,median_days_to_modification,5714,3.067,2.8915,3.594


,year,n_30d,n_repos,change_rate_30d,ci_30d_lower,ci_30d_upper,change_rate_30d_no_alima,no_alima_lower,no_alima_upper
0,2022,53,43,0.396226,0.259259,0.555556,0.396226,0.258621,0.551020
1,2023,1376,999,0.304506,0.273425,0.335983,0.304506,0.272341,0.334918
2,2024,3643,2438,0.236893,0.213796,0.258951,0.236893,0.213693,0.259604
3,2025,10451,5737,0.219118,0.200209,0.236575,0.212834,0.197737,0.229281
4,2026,4031,2166,0.224262,0.196132,0.251172,0.211490,0.191758,0.232300
